# Lecture 3: Seasonal Storage Notebook

This notebook explores the large-scale integration of renewable generation in the power system, the resulting supply-demand mismatch, and the role of seasonal storage.

The workflow is organized as follows:
1. Prepare German load, solar, and wind time series.
2. Analyze residual load under increasing renewable penetration.
3. Estimate ideal and non-ideal storage requirements.
4. Decompose storage demand into long-, mid-, and short-term components.
5. Derive comparable key performance indicators (KPIs) for storage energy and power.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
pd.options.plotting.backend = "plotly"

PLOT_TEMPLATE = "plotly_white"
# PLOT_TEMPLATE = "plotly_dark"

TIMEZONE = "Europe/Berlin"
DATA_PATH = "../data/time_series_60min_singleindex_filtered.csv"

HOURS_PER_STEP = 1
MW_TO_GW = 1e-3
MWH_TO_TWH = 1e-6
TWH_TO_MWH = 1e6

## 0. Data and helper functions

We again use the [Open Power System Dataset](https://data.open-power-system-data.org/time_series) as the basis for this toy model. The first step is to load the dataset and keep only the German demand, solar, and wind time series required for the analysis.

In [ ]:
def load_timeseries_data(path, timezone=TIMEZONE):
    """Load the OPSD time series and convert the index to local time."""
    data = pd.read_csv(path, index_col=0, parse_dates=True)
    data.index = data.index.tz_convert(timezone)
    return data


data = load_timeseries_data(DATA_PATH)

In [ ]:
def build_tso_dataframe(profile, tso):
    """Return a clean dataframe with load, solar, and aggregated wind generation."""
    wind_columns = [
        f"{tso}_wind_onshore_generation_actual",
        f"{tso}_wind_offshore_generation_actual",
    ]
    wind_columns = profile.columns.intersection(wind_columns)

    df = pd.concat(
        [
            profile[f"{tso}_load_actual_entsoe_transparency"].rename("load"),
            profile[f"{tso}_solar_generation_actual"].rename("solar"),
            profile[wind_columns].sum(axis=1).rename("wind"),
        ],
        axis=1,
    )
    return df

In [ ]:
df_DE = build_tso_dataframe(data, "DE")

In [ ]:
df_DE.head()

## 1. Residual load with increasing renewable penetration

To decarbonize the electricity sector, Germany and many other countries continue to expand solar and wind power. This notebook uses Germany in the base year 2019 and explores how additional renewable capacity changes the residual load and the implied storage requirement.

The question is deliberately simple: under the assumption of no additional imports or exports, how much renewable overbuild and how much storage would be required to balance the system?

### Residual load

Just as in the previous notebook, let's take a look at the residual load $P_{res}$ for Germany in the year 2019. This gives us an insight on the current state of renewable technologies' integration in our system. For the year 2019, Germany completed an installed capacity of solar PV of around 30 GW and a wind-power installed capacity of about 45 GW, while its peak load reached over 76 GW.

In [ ]:
max_load = df_DE["load"].max() / 1e3
max_solar = df_DE["solar"].max() / 1e3
max_wind = df_DE["wind"].max() / 1e3
share = (df_DE["solar"].sum() + df_DE["wind"].sum()) / df_DE["load"].sum() * 100

print("Germany 2019:")
print("-------------")
print(f"Peak load:                {max_load:.2f} GW")
print(f"Solar installed capacity: {max_solar:.2f} GW")
print(f"Wind installed capacity:  {max_wind:.2f} GW")
print(f"Share of renewables:      {share:.2f} %") # Share of renewables in gross electricity consumption

Renewable penetration has increased substantially since 2019. The 2019 dataset is therefore best interpreted as a transparent reference case for illustrating the methodology rather than as a representation of the current system state.

In [ ]:
def add_residual_load(df):
    """Return a copy with residual load added as load - solar - wind."""
    df_out = df.copy()
    df_out["residual"] = df_out["load"] - df_out["solar"] - df_out["wind"]
    return df_out

In [ ]:
# Add the residual load to the dataframe
df_DE = add_residual_load(df_DE)

### Residual load under renewable scaling

The next step is to quantify how the residual-load duration curve shifts when wind and solar generation are scaled by a common factor.

In [ ]:
def scale_renewables(df, scales, include_base=True):
    """Build residual-load scenarios for different renewable scaling factors."""
    scenarios = {}

    if include_base:
        scenarios["residual: base year"] = df["residual"]

    for scale in scales:
        df_scaled = df[["load", "solar", "wind"]].copy()
        df_scaled["solar"] *= scale
        df_scaled["wind"] *= scale
        df_scaled = add_residual_load(df_scaled)
        scenarios[f"residual: scale x{scale:g}"] = df_scaled["residual"]

    return pd.DataFrame(scenarios, index=df.index)

In [ ]:
df_scaled = scale_renewables(df_DE, np.arange(2.0, 6.0))

In [ ]:
def plot_residual_curves(df, **kwargs):
    """Plot residual-load duration curves for a set of scenarios."""
    fig = go.Figure()
    fig.update_layout(xaxis_title="Time [h]", yaxis_title="Power [MW]", **kwargs)
    fig.add_trace(
        go.Scatter(
            x=np.arange(df.shape[0]),
            y=np.zeros(df.shape[0]),
            line={"color": "grey", "dash": "dash"},
            opacity=0.7,
            showlegend=False,
            name="zero line",
        )
    )

    for name, residual in df.items():
        sorted_residual = residual.sort_values(ascending=False).values
        fig.add_trace(go.Scatter(x=np.arange(sorted_residual.size), y=sorted_residual, name=name))

    return fig

In [ ]:
plot_residual_curves(df_scaled, title="Residual-load duration curve with increasing renewable penetration", width=800, height=600, template=PLOT_TEMPLATE)

### Generation-consumption balance

The residual-load duration curves already show long periods of negative residual load for large renewable scaling factors. The following step translates that observation into annual deficit and surplus energy.

In [ ]:
df_scaled = scale_renewables(df_DE, np.arange(1.5, 6.0, 0.5))

In [ ]:
pd.concat([
    df_scaled.sum().rename("Energy Balance [TWh]") / 1e6, 
    df_scaled.min().rename("Min. Power [GW]") / 1e3,
    df_scaled.max().rename("Max. Power [GW]") / 1e3
    ], 
    axis=1
)

In [ ]:
def calculate_energy_balance(df_residual):
    """Split residual energy into annual demand deficit and renewable surplus."""
    deficit_twh = df_residual.clip(lower=0).sum() * MWH_TO_TWH
    surplus_twh = df_residual.clip(upper=0).sum() * MWH_TO_TWH

    return pd.DataFrame({
        "demand": deficit_twh,
        "surplus": surplus_twh,
    })

In [ ]:
calculate_energy_balance(df_scaled).plot.bar(template=PLOT_TEMPLATE, labels={"value": "Energy [TWh]", "index": "Scenario"}, title="Energy balance with increasing renewable penetration")

In the reference year 2019, the residual load remains positive throughout the year, so no renewable surplus occurs. As the renewable generation is scaled up, the annual surplus increases while the annual demand deficit decreases. Even in the most aggressive scenario shown here (`scale = 5.5`), renewable generation alone does not cover demand at every hour of the year, despite a large annual surplus.

## 2. Balancing residual load with bulk storage

We now introduce energy storage to achieve a fully renewable electricity supply. The goal is to estimate how much renewable overbuild is required and how large the storage system has to be.

In [ ]:
# Scale renewable generation until annual renewable energy matches annual demand
renewable_share = (df_DE["solar"].sum() + df_DE["wind"].sum()) / df_DE["load"].sum()
scale = 1 / renewable_share
print(f"Required renewable scaling for annual energy balance: {scale:.2f}x")

### Ideal storage with no losses

In [ ]:
def build_ideal_storage_case(df, renewable_scale):
    """Create the ideal storage case with unlimited power, unlimited energy, and no losses."""
    df_case = df.copy()
    df_case["solar"] *= renewable_scale
    df_case["wind"] *= renewable_scale
    df_case = add_residual_load(df_case)
    df_case["storage_power"] = -df_case["residual"]
    df_case["storage_energy_mwh"] = df_case["storage_power"].cumsum() * HOURS_PER_STEP
    return df_case


df_ideal_storage = build_ideal_storage_case(df_DE, scale)

In [ ]:
df_ideal_storage["storage_power"].plot(template=PLOT_TEMPLATE, labels={"value": "Power [MW]"}, title="Ideal storage power")

In [ ]:
df_ideal_storage["storage_energy_mwh"].plot(template=PLOT_TEMPLATE, labels={"value": "Energy [MWh]"}, title="Ideal storage energy level")

In [ ]:
ideal_storage_cap_twh = (
    df_ideal_storage["storage_energy_mwh"].max() - df_ideal_storage["storage_energy_mwh"].min()
) * MWH_TO_TWH
print(f"Ideal storage capacity: {ideal_storage_cap_twh:.2f} TWh")

### Non-ideal storage with limited energy, power, and efficiency

This section simulates a generic storage system with finite energy capacity, finite converter power, and charge/discharge losses.

In [ ]:
def simulate_bucket_storage_operation(
    df,
    storage_capacity_mwh,
    max_storage_power_mw,
    eta_charge,
    eta_discharge,
    initial_soe_fraction=0.5,
):
    """Simulate storage operation for an hourly residual-load time series."""
    result = df.copy()

    residual_values = result["residual"].to_numpy(dtype=float)
    energy = np.zeros(len(result), dtype=float)
    power_dc = np.zeros(len(result), dtype=float)
    power_ac = np.zeros(len(result), dtype=float)

    soe = storage_capacity_mwh * initial_soe_fraction

    for idx, residual in enumerate(residual_values):
        if residual > 0:
            requested_power_dc = residual / eta_discharge
        else:
            requested_power_dc = max(residual * eta_charge, -max_storage_power_mw)

        next_soe = np.clip(soe - requested_power_dc * HOURS_PER_STEP, 0, storage_capacity_mwh)
        actual_power_dc = (next_soe - soe) / HOURS_PER_STEP

        if residual > 0:
            actual_power_ac = actual_power_dc * eta_discharge
        else:
            actual_power_ac = actual_power_dc / eta_charge

        energy[idx] = next_soe
        power_dc[idx] = actual_power_dc
        power_ac[idx] = actual_power_ac
        soe = next_soe

    result["energy"] = energy
    result["power_dc"] = power_dc
    result["power_ac"] = power_ac
    return result

In [ ]:
renewable_scale = 3.7          # Renewable overbuild to compensate storage losses
max_storage_power_mw = 80e3   # 80 GW
storage_capacity_mwh = 20e6   # 20 TWh
eta_charge = 0.6
eta_discharge = 0.6

df_storage_case = df_DE.copy()
for col in ["solar", "wind"]:
    df_storage_case[col] = pd.to_numeric(df_storage_case[col], errors="coerce").astype(float)

df_storage_case["solar"] *= renewable_scale
df_storage_case["wind"] *= renewable_scale

df_storage_case = add_residual_load(df_storage_case)
df_storage_case = simulate_bucket_storage_operation(
    df_storage_case,
    storage_capacity_mwh=storage_capacity_mwh,
    max_storage_power_mw=max_storage_power_mw,
    eta_charge=eta_charge,
    eta_discharge=eta_discharge,
)

df_storage_case["storage_power"] = -df_storage_case["power_ac"]

In [ ]:
df_storage_case["power_dc"].plot(template=PLOT_TEMPLATE, labels={"value": "Power [MW]"}, title="Storage-side converter power")

In [ ]:
df_storage_case["energy"].plot(template=PLOT_TEMPLATE, labels={"value": "Energy [MWh]"}, title="Storage state of energy")

In [ ]:
# Compare residual load and actual storage power seen from the grid
df_storage_case[["residual", "storage_power"]].plot(
    template=PLOT_TEMPLATE,
    labels={"value": "Power [MW]"},
    title="Residual load and delivered storage power",
)

In [ ]:
# Fraction of positive residual energy that is covered by storage discharge
residual_demand = df_storage_case["residual"].clip(lower=0)
served_by_storage = pd.concat(
    [df_storage_case["storage_power"].clip(lower=0), residual_demand],
    axis=1,
).min(axis=1)
fulfillment = served_by_storage.sum() / residual_demand.sum()
print(f"Residual-load coverage factor: {fulfillment * 100:.2f} %")

In [ ]:
# Ratio between annual demand deficit and annual renewable surplus before storage
balance = (
    df_storage_case.loc[df_storage_case["residual"] > 0, "residual"].sum()
    / df_storage_case.loc[df_storage_case["residual"] < 0, "residual"].abs().sum()
)
print(f"Demand-to-surplus energy ratio: {balance * 100:.2f} %")

In [ ]:
def calculate_power_utilization(power_series):
    """Mean absolute power normalized by the rated storage power."""
    rated_power = power_series.abs().max()
    if pd.isna(rated_power) or rated_power == 0:
        return np.nan
    return power_series.abs().dropna().mean() / rated_power


tau_P = calculate_power_utilization(df_storage_case["storage_power"])
print(f"Power-based utilization tau_P: {tau_P:.2f}")

## 3. Storage-operation analysis and hybrid storage concepts

There is no single storage technology that is optimal for every time scale. A realistic system is more likely to combine different technologies with different power, efficiency, and cost characteristics.

Here, we approximate such a hybrid storage system by decomposing the ideal storage trajectory into three time scales:
- Long-term storage (weekly trend)
- Mid-term storage (daily trend)
- Short-term storage (intra-day residual)

The objective is to compare energy capacity, power requirement, throughput, and power-based utilization across these abstract storage classes.

In [ ]:
def decompose_storage_timescales(df, long_window="7D", mid_window="1D"):
    """Decompose total storage energy into long-, mid-, and short-term components."""
    df_out = df[["residual"]].copy()
    df_out["energy_total_twh"] = df_out["residual"].cumsum() * MWH_TO_TWH

    df_out["long_term_storage"] = df_out["energy_total_twh"].rolling(pd.Timedelta(long_window)).mean()
    df_out["energy_minus_long"] = df_out["energy_total_twh"] - df_out["long_term_storage"]

    df_out["mid_term_storage"] = df_out["energy_minus_long"].rolling(pd.Timedelta(mid_window)).mean()
    df_out["energy_minus_long_mid"] = df_out["energy_minus_long"] - df_out["mid_term_storage"]

    df_out["short_term_storage"] = (
        df_out["energy_total_twh"]
        - df_out["long_term_storage"]
        - df_out["mid_term_storage"]
    )
    df_out["reconstructed_energy_twh"] = (
        df_out["long_term_storage"]
        + df_out["mid_term_storage"]
        + df_out["short_term_storage"]
    )
    return df_out


df_hybrid = decompose_storage_timescales(df_ideal_storage)

In [ ]:
df_hybrid[["energy_total_twh", "long_term_storage"]].plot(template=PLOT_TEMPLATE, labels={"value": "Energy [TWh]"}, title="Total storage energy and long-term component")

In [ ]:
# The decomposition keeps the intermediate components for transparency
# and makes it easier to inspect the effect of each smoothing step.
df_hybrid[[
    "energy_total_twh",
    "energy_minus_long",
    "mid_term_storage",
    "energy_minus_long_mid",
    "short_term_storage",
]].head()

In [ ]:
df_hybrid[["energy_total_twh", "reconstructed_energy_twh"]].plot(template=PLOT_TEMPLATE, labels={"value": "Energy [TWh]"}, title="Check of hybrid-storage reconstruction")

In [ ]:
df_hybrid[["short_term_storage", "mid_term_storage", "long_term_storage"]].plot(
    template=PLOT_TEMPLATE,
    labels={"value": "Energy [TWh]"},
    title="Decomposed storage energy by time scale",
)

In [ ]:
def calculate_storage_power_timeseries(df_energy, columns):
    """Convert hourly energy changes from TWh to MW."""
    return df_energy[columns].diff() * TWH_TO_MWH


def calculate_storage_kpis(df_energy, columns):
    """Return comparable energy and power KPIs for the selected storage states."""
    df_power = calculate_storage_power_timeseries(df_energy, columns)
    capacity_need_twh = df_energy[columns].abs().max()
    energy_throughput_twh = df_energy[columns].diff().abs().sum()
    pmax_storage_mw = df_power.abs().max()
    tau_p = pd.Series(
        {column: calculate_power_utilization(df_power[column]) for column in columns},
        name="tau_P",
    )
    return capacity_need_twh, energy_throughput_twh, pmax_storage_mw, df_power, tau_p


storage_state_cols = [
    "energy_total_twh",
    "short_term_storage",
    "mid_term_storage",
    "long_term_storage",
]

(
    capacity_need_twh,
    energy_throughput_twh,
    pmax_storage_mw,
    df_power_mw,
    tau_P_by_storage,
) = calculate_storage_kpis(df_hybrid, storage_state_cols)

In [ ]:
hybrid_storage_summary = pd.concat(
    [
        capacity_need_twh.rename("capacity_need_twh"),
        energy_throughput_twh.rename("energy_throughput_twh"),
        pmax_storage_mw.rename("pmax_storage_mw"),
        tau_P_by_storage,
    ],
    axis=1,
)

hybrid_storage_summary

In [ ]:
# Peak converter power requirement per storage class
pmax_storage_mw

In [ ]:
# Storage power time series derived from the energy trajectories
df_power_mw.plot(template=PLOT_TEMPLATE, labels={"value": "Power [MW]"}, title="Storage power by time scale")

In [ ]:
# Power-based utilization of the hybrid-storage components
for tech, value in tau_P_by_storage.drop(index="energy_total_twh").items():
    print(f"{tech}: {value:.2f}")

### Interpretation of the power-based utilization $\tau_P$

The metric `tau_P` is defined analogously to the capacity factor of a power plant:

$$
\tau_P = \frac{\overline{|P_{storage}|}}{P_{storage,max}}
$$

- `0` means the storage power electronics are barely used.
- `1` would mean the storage operates continuously at rated charge/discharge power.
- This is a **power-based** indicator and must therefore be normalized by **power**, not by energy capacity.
- The same definition can be applied consistently to the full storage trajectory and to each hybrid-storage component.

### Optional exploratory analysis

The following cells provide additional diagnostics on the distribution of storage energy, storage power, and event durations for the three abstract storage classes.

In [ ]:
# Distribution of the storage-energy levels
df_hybrid[["long_term_storage", "mid_term_storage", "short_term_storage"]].plot.hist(
    labels={"value": "Energy [TWh]"}
).update_layout(barmode="overlay").update_traces(opacity=0.75)

In [ ]:
# Distribution of storage power
df_power_mw[["long_term_storage", "mid_term_storage", "short_term_storage"]].plot.hist(
    labels={"value": "Power [MW]"}
).update_layout(barmode="overlay").update_traces(opacity=0.75)

In [ ]:
def group_nonzero_events(series):
    """Group contiguous positive-power events in a time series."""
    series = series.fillna(0)
    active = series > 0
    event_id = active.ne(active.shift(fill_value=False)).cumsum().where(active, 0)
    return series.to_frame(name=series.name).assign(event_id=event_id).groupby("event_id")

In [ ]:
groups = {}
for name, series in df_power_mw[["long_term_storage", "mid_term_storage", "short_term_storage"]].items():
    groups[name] = group_nonzero_events(series.clip(lower=0).rename(name))

In [ ]:
# Event energy
sums = pd.DataFrame()
for name, group in groups.items():
    group_sum = group.sum(numeric_only=True)[name]
    group_sum = group_sum.loc[group_sum.index != 0]
    group_sum = group_sum.sort_values(ascending=False).reset_index(drop=True)
    sums = pd.concat([sums, group_sum], axis=1)

In [ ]:
sums.plot.bar(facet_row="variable", height=600, labels={"value": "Energy [MWh]", "index": "Event rank"})

In [ ]:
# Event duration
durations = pd.DataFrame()
for name, group in groups.items():
    group_count = group.count()[name]
    group_count = group_count.loc[group_count.index != 0]
    group_count = group_count.sort_values(ascending=False).reset_index(drop=True)
    durations = pd.concat([durations, group_count], axis=1)

In [ ]:
durations.plot.bar(facet_row="variable", height=600, labels={"value": "Duration [h]", "index": "Event rank"})